# 02_Analysis.ipynb — Analysis
Computes sinuosity, slope, segment means, flux, fits beta'1 (slope profile)
and beta'2 (elevation profile) with SE, plus goodness-of-fit for both.

In [44]:
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats as scipy_stats


## Load data

In [45]:
with open("data.pkl", "rb") as f:
    data = pickle.load(f)
stream_registry = data["stream_registry"]
ALPHA           = data["ALPHA"]
print(f"Loaded {len(stream_registry)} streams")


Loaded 16 streams


## Step 1 — Sinuosity (arc & chord length per segment) 
   #### Refer Supplementary Section 1

In [46]:
def compute_segment_stats(df):
    rows = []
    for seg_id, g in df.groupby("segment_id"):
        if len(g) < 2:
            continue
        xy          = g[["x", "y"]].values
        sinuous_len = np.sum(np.linalg.norm(np.diff(xy, axis=0), axis=1))
        normal_len  = np.linalg.norm(xy[-1] - xy[0])
        rows.append([seg_id, normal_len, sinuous_len])

    ss = pd.DataFrame(rows, columns=["segment_id", "normal_length", "sinuous_length"])
    ss["sinuosity"] = ss["sinuous_length"] / ss["normal_length"]
    ss["Acc_Norm"]  = ss["normal_length"].cumsum()
    ss["Acc_Sin"]   = ss["sinuous_length"].cumsum()
    return ss

for entry in stream_registry:
    df = entry[0]
    df.attrs["seg_stats"] = compute_segment_stats(df)

print("Sinuosity computed.")


Sinuosity computed.


## Step 2 — Segment means (velocity, elevation, relief, x, y, distance)
   ### Refer supplementary text S1

In [47]:
def compute_segment_means(df):
    vc     = df.attrs.get("vel_col",  "velocity")
    ec     = df.attrs.get("elev_col", "elev")
    offset = df.attrs.get("elev_offset", 0)

    agg = (
        df.groupby("segment_id")
          .agg(relief         = ("relief",   "mean"),
               velocity       = (vc,         "mean"),
               x              = ("x",        "mean"),
               y              = ("y",        "mean"),
               Distance       = ("dist_N",   "mean"),
               elevation      = (ec,         "mean"),
               elevation_max  = (ec,         "max"))
          .reset_index()
    )

    agg["elevation"] = agg["elevation"] + offset

    return agg

for entry in stream_registry:
    df = entry[0]
    df.attrs["seg_means"] = compute_segment_means(df)

print("Segment means computed.")

Segment means computed.


## Step 3 — Slope per segment

#### See supplementtary text S1

In [48]:
def compute_slope(df, seg_stats, seg_means, elev_col="elev", n=1):
    # trim to matching length to avoid size mismatch
    n_rows = min(len(seg_stats), len(seg_means))
    ss     = seg_stats.iloc[:n_rows].copy().reset_index(drop=True)
    sm     = seg_means.iloc[:n_rows].copy().reset_index(drop=True)

    # slope_l: elevation difference between neighbouring segments
    elev    = sm["elevation"].values
    sin = sm["Distance"].values

    delta_elev    = np.diff(elev)
    delta_sin     = np.diff(sin)

    with np.errstate(divide="ignore", invalid="ignore"):
        slope_l = -delta_elev / delta_sin

    # first segment has no previous neighbour — pad with NaN
    slope_l = np.concatenate([[np.nan], slope_l])
    ss["slope_l"] = slope_l

    # slope_d: delta_z / normal_length per segment
    ec = df.attrs.get("elev_col", elev_col)
    delta_z = {}
    for seg_id, seg in df.groupby("segment_id"):
        z  = seg[ec].values
        dz = np.mean(z[:n]) - np.mean(z[-n:]) if len(z) >= n else z[0] - z[-1]
        delta_z[seg_id] = dz

    ss["delta_z"] = ss["segment_id"].map(delta_z)
    ss["slope_d"] = -(ss["delta_z"] / ss["normal_length"])
    return ss

for entry in stream_registry:
    df = entry[0]
    ec = df.attrs.get("elev_col", "elev")
    df.attrs["seg_stats"] = compute_slope(df, df.attrs["seg_stats"], df.attrs["seg_means"],
                                          elev_col=ec, n=1)

print("Slopes computed.")

Slopes computed.


## Step 4 — Merge segment means into segment stats and compute flux

In [49]:
def compute_flux(seg_stats, seg_means):
    ss = seg_stats.merge(
        seg_means[["segment_id", "velocity", "elevation", "relief",
                   "x", "y", "Distance", "elevation_max"]],
        on="segment_id", how="left"
    )
    ss["flux"]   = ss["sinuosity"] * ss["velocity"]
    ss["flux_1"] = ss["sinuosity"] * ss["velocity"] / ss["normal_length"]
    ss["flux_d"] = ss["flux_1"].diff().fillna(ss["flux_1"])
    return ss

for entry in stream_registry:
    df = entry[0]
    df.attrs["seg_stats"] = compute_flux(df.attrs["seg_stats"], df.attrs["seg_means"])

print("Flux computed and merged into seg_stats.")


Flux computed and merged into seg_stats.


## Step 5 — Remove last segment row (Stream length <500/2000m)

In [50]:
for entry in stream_registry:
    df, group, label, *_ = entry
    df.attrs["seg_stats"] = df.attrs["seg_stats"].iloc[:-1].reset_index(drop=True)

print("Last row removed.")

Last row removed.


## Step 6 — Find beta'1 from slope profile
   #### Refer Eq. (3)

In [51]:
def fit_gamma1(x, slope, alpha):
    """Fit beta'_1 from slope profile S = beta'_1 * x^((alpha-1)/2)."""
    exp = (alpha - 1) / 2

    if alpha == 1:           # constant: beta'_1 = mean(S)
        b1   = np.mean(slope)
        n    = len(slope)
        se   = np.std(slope, ddof=1) / np.sqrt(n)
        pred = np.full_like(slope, b1, dtype=float)
    else:                    # power-law through origin
        X    = x ** exp
        b1   = np.sum(X * slope) / np.sum(X ** 2)
        resid = slope - b1 * X
        n    = len(slope)
        se   = np.sqrt(np.sum(resid**2) / ((n - 1) * np.sum(X**2)))
        pred = b1 * X

    return b1, se, pred

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    alpha = ALPHA[trend]
    ss    = df.attrs["seg_stats"].dropna(subset=["slope_l"])
    x, sl = ss["Distance"].values, ss["slope_l"].values

    g1, se1, sl_pred = fit_gamma1(x, sl, alpha)
    df.attrs["beta'1"]     = g1
    df.attrs["SE_beta'1"]  = se1
    df.attrs["slope_pred"] = sl_pred
    df.attrs["slope_x"]    = x
    df.attrs["slope_obs"]  = sl
    df.attrs["alpha"]      = alpha

print(f"{'Stream':18s}  {"beta'_1":>12s}  {"SE_beta'1":>12s}")
print("-" * 48)
for entry in stream_registry:
    df, group, label, *_ = entry
    print(f"{label:18s}  {df.attrs["beta'1"]:12.4g}  {df.attrs["SE_beta'1"]:12.4g}")


Stream                   beta'_1     SE_beta'1
------------------------------------------------
sia_1                      1.383        0.1779
sia_2                    0.01959      0.001023
sia_3                      1.492        0.1237
sia_4                    0.01886      0.001049
sia_5                    0.01574     0.0007579
sia_6                      1.533        0.1169
ron_1                      2.902         0.195
fou_1                  0.0009635     6.007e-05
fou_2                   0.001098     7.907e-05
fou_3                   0.001375      0.000177
fou_4                   0.001623     0.0001688
von_1                  0.0005734     2.549e-05
von_2                  0.0006366     2.388e-05
von_3                  0.0007478     3.724e-05
von_4                  0.0006025     2.769e-05
vet_1                    0.02299       0.00102


## Step 7 — Find beta'2 from elevation profile
   #### Refer Eq. (4)

In [52]:
def fit_beta2(x, z, alpha):
    """Fit beta'_2 from elevation profile."""
    exp = (alpha + 1) / 2
    x0, z0 = x[0], z[0]
    F  = x**exp - x0**exp
    Y  = z0 - z

    k  = np.sum(F * Y) / np.sum(F**2)
    b2 = k * exp

    resid = Y - k * F
    n     = len(Y)
    se_k  = np.sqrt(np.sum(resid**2) / ((n - 1) * np.sum(F**2)))
    se_b2 = se_k * exp

    z_pred = z0 - k * F
    return b2, se_b2, z_pred

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    alpha = ALPHA[trend]
    ss    = df.attrs["seg_stats"]
    x     = ss["Distance"].values
    z     = ss["elevation"].values   # already merged into seg_stats

    b2, se2, z_pred = fit_beta2(x, z, alpha)

    # Veteranbreen: correct z_pred values below 0 (offset artifact)
    if group == "Veteranbreen":
        z_pred = np.where(z_pred < 0, z_pred + 2000, z_pred)

    df.attrs["beta'2"]    = b2
    df.attrs["SE_beta'2"] = se2
    df.attrs["z_pred"]    = z_pred
    df.attrs["elev_x"]    = x
    df.attrs["elev_obs"]  = z

print(f"{'Stream':18s}  {"beta'2":>12s}  {"SE_beta'2":>12s}")
print("-" * 48)
for entry in stream_registry:
    df, group, label, *_ = entry
    print(f"{label:18s}  {df.attrs["beta'2"]:12.4g}  {df.attrs["SE_beta'2"]:12.4g}")


Stream                    beta'2     SE_beta'2
------------------------------------------------
sia_1                       1.21        0.0498
sia_2                    0.01927     0.0001698
sia_3                      1.291       0.04236
sia_4                    0.01837     9.311e-05
sia_5                    0.01567     9.759e-05
sia_6                      1.381       0.02196
ron_1                      2.506       0.02742
fou_1                   0.001034     1.571e-05
fou_2                   0.001279     3.846e-05
fou_3                   0.001689     4.965e-05
fou_4                   0.001781     5.314e-05
von_1                   0.000648     8.922e-06
von_2                  0.0006479     5.523e-06
von_3                  0.0008513     1.435e-05
von_4                   0.000641      7.72e-06
vet_1                    0.02201     0.0002678


## Step 8 — Goodness-of-fit: slope profile (beta'1)
 #### See Supplementary Table S3

In [53]:
print("\n=== Slope Profile Fit (beta'1) — per stream ===")

print(f"{'Stream':18s}  {'R²':>7s}  {'Adj R²':>8s}  "
      f"{'RMSE':>10s}  {'NRMSE(M)':>10s}  {'MAE':>10s}  {'Pearson r':>10s}")

print("-" * 90)

all_obs_s, all_pred_s = [], []
k = 1  # number of predictors in the slope fit

for entry in stream_registry:

    df, group, label, *_ = entry

    obs  = df.attrs["slope_obs"]
    pred = df.attrs["slope_pred"]

    n = min(len(obs), len(pred))

    o, p = obs[:n], pred[:n]

    mask = np.isfinite(o) & np.isfinite(p)
    o, p = o[mask], p[mask]

    if len(o) < 2:
        continue

    r2     = r2_score(o, p)

    adj_r2 = 1 - (1 - r2) * (len(o) - 1) / (len(o) - k - 1)

    rmse   = np.sqrt(mean_squared_error(o, p))

    # NRMSE
    nrmse_M = rmse / (np.mean(o))
    

    mae    = mean_absolute_error(o, p)

    r, _   = scipy_stats.pearsonr(o, p)

    print(f"{label:18s}  "
          f"{r2:7.4f}  "
          f"{adj_r2:8.4f}  "
          f"{rmse:10.6f}  "
          f"{nrmse_M:10.6f}  "
          f"{mae:10.6f}  "
          f"{r:10.4f}")

    all_obs_s.extend(o)
    all_pred_s.extend(p)

# ===== pooled statistics =====

ao, ap = np.array(all_obs_s), np.array(all_pred_s)

mask = np.isfinite(ao) & np.isfinite(ap)

ao, ap = ao[mask], ap[mask]

r2p = r2_score(ao, ap)

adj_r2p = 1 - (1 - r2p) * (len(ao) - 1) / (len(ao) - k - 1)

rmsep = np.sqrt(mean_squared_error(ao, ap))

# pooled NRMSE
nrmsep_M  = rmse / (np.mean(ao))

maep = mean_absolute_error(ao, ap)

rp, _ = scipy_stats.pearsonr(ao, ap)

print("-" * 90)

print(f"{'POOLED (all)':18s}  "
      f"{r2p:7.4f}  "
      f"{adj_r2p:8.4f}  "
      f"{rmsep:10.6f}  "
      f"{nrmsep_M:10.6f}  "
      f"{maep:10.6f}  "
      f"{rp:10.4f}")


=== Slope Profile Fit (beta'1) — per stream ===
Stream                   R²    Adj R²        RMSE    NRMSE(M)         MAE   Pearson r
------------------------------------------------------------------------------------------
sia_1               -0.5437   -1.0582    0.004714    0.254272    0.003745      0.3413
sia_2                0.0000   -0.0667    0.004093    0.208941    0.003483         nan
sia_3               -0.2853   -0.7138    0.003279    0.165262    0.002888      0.7243
sia_4                0.0000   -0.0500    0.004807    0.254864    0.003453         nan
sia_5                0.0000   -0.0476    0.003555    0.225869    0.002763         nan
sia_6                0.6331    0.5807    0.003705    0.224793    0.002970      0.7972
ron_1                0.7447    0.7083    0.006182    0.200986    0.005523      0.8658
fou_1                0.5326    0.4807    0.010831    0.201658    0.007365      0.7491
fou_2               -0.3899   -0.5885    0.011721    0.202572    0.008839      0.6099


/tmp/ipykernel_1379306/2364903542.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _   = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_1379306/2364903542.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _   = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_1379306/2364903542.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _   = scipy_stats.pearsonr(o, p)
/tmp/ipykernel_1379306/2364903542.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _   = scipy_stats.pearsonr(o, p)


## Step 9 — Goodness-of-fit: elevation profile (beta'2)
   #### See Supplementary Table S3

In [54]:
print("\n Elevation Profile Fit (beta'2) — per stream")

print(f"{'Stream':18s}  {'R²':>7s}  {'Adj R²':>8s}  "
      f"{'RMSE(m)':>10s}  {'NRMSE(M)':>10s}  "
      f"{'MAE(m)':>10s}  {'Pearson r':>10s}")

print("-" * 90)

all_obs_e, all_pred_e = [], []

k = 1  # one free parameter (beta'_2)

for entry in stream_registry:

    df, group, label, *_ = entry

    obs  = df.attrs["elev_obs"]
    pred = df.attrs["z_pred"]

    n = min(len(obs), len(pred))

    o, p = obs[:n], pred[:n]

    mask = np.isfinite(o) & np.isfinite(p)

    o, p = o[mask], p[mask]

    if len(o) < 2:
        continue

    r2 = r2_score(o, p)

    adj_r2 = 1 - (1 - r2) * (len(o) - 1) / (len(o) - k - 1)

    rmse = np.sqrt(mean_squared_error(o, p))

    # NRMSE
    nrmse_M  = rmse / (np.mean(o))

    mae = mean_absolute_error(o, p)

    r, _ = scipy_stats.pearsonr(o, p)

    print(f"{label:18s}  "
          f"{r2:7.4f}  "
          f"{adj_r2:8.4f}  "
          f"{rmse:10.2f}  "
          f"{nrmse_M:10.4f}  "
          f"{mae:10.2f}  "
          f"{r:10.4f}")

    all_obs_e.extend(o)
    all_pred_e.extend(p)

# pooled statistics

ao, ap = np.array(all_obs_e), np.array(all_pred_e)

mask = np.isfinite(ao) & np.isfinite(ap)

ao, ap = ao[mask], ap[mask]

r2p = r2_score(ao, ap)

adj_r2p = 1 - (1 - r2p) * (len(ao) - 1) / (len(ao) - k - 1)

rmsep = np.sqrt(mean_squared_error(ao, ap))

# pooled NRMSE
nrmsep_M = rmsep / (np.mean(ao))

maep = mean_absolute_error(ao, ap)

rp, _ = scipy_stats.pearsonr(ao, ap)

print("-" * 90)

print(f"{'POOLED (all)':18s}  "
      f"{r2p:7.4f}  "
      f"{adj_r2p:8.4f}  "
      f"{rmsep:10.2f}  "
      f"{nrmsep_M:10.4f}  "
      f"{maep:10.2f}  "
      f"{rp:10.4f}")


 Elevation Profile Fit (beta'2) — per stream
Stream                   R²    Adj R²     RMSE(m)    NRMSE(M)      MAE(m)   Pearson r
------------------------------------------------------------------------------------------
sia_1                0.9730    0.9662       10.80      0.0023        8.12      0.9911
sia_2                0.9953    0.9950       13.94      0.0031       10.90      0.9978
sia_3                0.9821    0.9777        9.20      0.0019        7.23      0.9942
sia_4                0.9978    0.9977       11.21      0.0025        8.70      0.9992
sia_5                0.9965    0.9963       12.57      0.0028        8.72      0.9988
sia_6                0.9903    0.9891        9.35      0.0020        8.02      0.9967
ron_1                0.9950    0.9944       11.66      0.0021        9.34      0.9976
fou_1                0.9924    0.9917        7.95      0.0150        6.33      0.9983
fou_2                0.9733    0.9700       13.34      0.0282       11.64      0.9957
fou

## Step 10 — Predicted sinuosity (using beta'1)
   #### Refer Section 5.2 Eq. (6)

In [55]:
for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    ss     = df.attrs["seg_stats"]
    x      = ss["Distance"].values
    slope  = ss["slope_d"].values
    alpha  = df.attrs["alpha"]
    beta1 = df.attrs["beta'1"]

    exp = (alpha - 1) / 2
    with np.errstate(divide="ignore", invalid="ignore"):
        sin_pred = np.where(x > 0, slope / (beta1 * x**exp), np.nan)
    df.attrs["sin_pred"] = sin_pred

print("Predicted sinuosity computed (using beta'_1).")


Predicted sinuosity computed (using beta'_1).


## Step 11 — Save analysis results

In [56]:
with open("analysis.pkl", "wb") as f:
    pickle.dump({"stream_registry": stream_registry, "ALPHA": ALPHA}, f)
print("Saved analysis.pkl")


Saved analysis.pkl


### Print raw dataframe, Segment_Means & Segment_stats

In [57]:
# show all rows and columns without truncation
pd.set_option("display.max_rows",    None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       1000)
pd.set_option("display.max_colwidth",None)


# change the label to whichever stream you want
target_label = "sia_1"

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    if label == target_label:
        print(f" raw dataframe for {label} ")
        print(df)
        print(f" seg_stats for {label} ")
        print(df.attrs["seg_stats"])
        print(f"\n seg_means for {label} ")
        print(df.attrs["seg_means"])
        break

 raw dataframe for sia_1 
      vertex_index      distance       angle      elev            x           y  Velocity  relief        dist_N  segment_id
0                0      0.000000    8.183272  4622.607 -1484712.571  785310.382   152.319  12.479  1.277790e+04           6
1                1      3.389307  333.825575  4622.697 -1484712.089  785313.736   152.319  13.634  1.277451e+04           6
2                2      7.149581  332.783638  4622.697 -1484715.363  785315.586   152.319  13.634  1.277075e+04           6
3                3      9.848806    6.099399  4622.697 -1484715.076  785318.270   152.319  13.634  1.276805e+04           6
4                4     12.548031    8.388944  4622.962 -1484714.789  785320.954   129.971  13.828  1.276535e+04           6
5                5     16.553374   10.678489  4622.962 -1484714.047  785324.890   129.971  13.828  1.276134e+04           6
6                6     20.558717  357.204332  4622.962 -1484713.305  785328.826   129.971  13.828  1.27573

In [58]:
# ── Mean sinuosity and mean slope_l per stream ──────────────────────
print(f"{'Stream':18s}  {'Mean Sinuosity':>15s}  {'Mean Slope_l':>13s}")
print("-" * 52)

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    ss = df.attrs["seg_stats"].dropna(subset=["sinuosity", "slope_l"])

    mean_sin   = ss["sinuosity"].mean()
    mean_slope = ss["slope_l"].mean()

    print(f"{label:18s}  {mean_sin:15.4f}  {mean_slope:13.6f}")

Stream               Mean Sinuosity   Mean Slope_l
----------------------------------------------------
sia_1                        1.0808       0.018541
sia_2                        1.2242       0.019588
sia_3                        1.0618       0.019841
sia_4                        1.3958       0.018860
sia_5                        1.5511       0.015739
sia_6                        1.3050       0.016483
ron_1                        1.5798       0.030759
fou_1                        1.1714       0.053712
fou_2                        1.4103       0.057859
fou_3                        1.0618       0.063466
fou_4                        1.1189       0.071593
von_1                        1.2378       0.037958
von_2                        1.1018       0.042066
von_3                        1.1076       0.043898
von_4                        1.0483       0.040489
vet_1                        1.0699       0.022992
